# C Implied-Volatility Solver

This notebook compiles and validates a C bisection solver for implied volatility. The C solver uses the same Black-Scholes pricing kernel as the C pricing notebook and returns explicit success or failure codes.

## Solver structure

The implied-volatility problem solves:

\[
BS(\sigma) - \text{market price} = 0
\]

Bisection starts with a lower and upper volatility bound. At each step, the midpoint is tested and the interval is cut in half. The loop stops when the pricing error is within tolerance or the interval is small enough.

In [ ]:
from pathlib import Path
import platform
import shutil
import subprocess
import sys
import timeit

import pandas as pd

In [ ]:
project_root = Path.cwd()

if not (project_root / "c_src").exists():
    project_root = project_root.parent

sys.path.insert(0, str(project_root))

## Compile the C library

The implied-volatility solver is compiled together with the normal-distribution helpers and Black-Scholes pricing functions.

In [ ]:
compiled_dir = project_root / "compiled"
compiled_dir.mkdir(exist_ok=True)

compiler = shutil.which("gcc") or shutil.which("clang")

if compiler is None:
    raise RuntimeError("No C compiler found. Install gcc or clang before running this notebook.")

system_name = platform.system().lower()

source_files = [
    project_root / "c_src" / "normal_math.c",
    project_root / "c_src" / "bs_pricer.c",
    project_root / "c_src" / "iv_solver.c",
]

if system_name == "darwin":
    library_path = compiled_dir / "libbs_pricer.dylib"
    compile_command = [
        compiler,
        "-O3",
        "-fPIC",
        "-dynamiclib",
        *map(str, source_files),
        "-o",
        str(library_path),
    ]
elif system_name == "windows":
    library_path = compiled_dir / "bs_pricer.dll"
    compile_command = [
        compiler,
        "-O3",
        "-shared",
        *map(str, source_files),
        "-o",
        str(library_path),
    ]
else:
    library_path = compiled_dir / "libbs_pricer.so"
    compile_command = [
        compiler,
        "-O3",
        "-fPIC",
        "-shared",
        *map(str, source_files),
        "-o",
        str(library_path),
        "-lm",
    ]

subprocess.run(compile_command, check=True)
library_path

## Load C and Python solvers

In [ ]:
from src.black_scholes import call_price, put_price
from src.c_bindings import BlackScholesCLibrary, safe_c_implied_vol_bisection
from src.implied_vol import solve_implied_volatility

bs_c = BlackScholesCLibrary(library_path)

## Python versus C IV comparison

In [ ]:
sample_options = pd.DataFrame(
    [
        {"option_type": "call", "S": 100.0, "K": 100.0, "T": 0.50, "r": 0.04, "sigma_true": 0.25},
        {"option_type": "put", "S": 100.0, "K": 100.0, "T": 0.50, "r": 0.04, "sigma_true": 0.30},
        {"option_type": "call", "S": 110.0, "K": 100.0, "T": 1.00, "r": 0.03, "sigma_true": 0.22},
        {"option_type": "put", "S": 95.0, "K": 100.0, "T": 45 / 365, "r": 0.04, "sigma_true": 0.35},
    ]
)

rows = []

for _, row in sample_options.iterrows():
    if row["option_type"] == "call":
        market_price = call_price(row["S"], row["K"], row["T"], row["r"], row["sigma_true"])
    else:
        market_price = put_price(row["S"], row["K"], row["T"], row["r"], row["sigma_true"])

    py_result = solve_implied_volatility(
        market_price,
        row["S"],
        row["K"],
        row["T"],
        row["r"],
        row["option_type"],
    )

    c_result = bs_c.implied_vol_bisection(
        market_price,
        row["S"],
        row["K"],
        row["T"],
        row["r"],
        row["option_type"],
    )

    rows.append(
        {
            "option_type": row["option_type"],
            "S": row["S"],
            "K": row["K"],
            "T": row["T"],
            "market_price": market_price,
            "true_sigma": row["sigma_true"],
            "python_iv": py_result.implied_volatility,
            "c_iv": c_result.implied_volatility,
            "absolute_difference": abs(py_result.implied_volatility - c_result.implied_volatility),
            "c_solver_status": c_result.solver_status,
            "c_failure_reason": c_result.failure_reason,
            "c_iterations": c_result.iterations,
        }
    )

comparison = pd.DataFrame(rows)
comparison.round(10)

## Failure cases

Failure flags make the output auditable. A failed solve is not silently converted into a misleading volatility value.

In [ ]:
failure_cases = pd.DataFrame(
    [
        {"case": "zero price", "market_price": 0.0, "S": 100.0, "K": 100.0, "T": 0.5, "r": 0.04, "option_type": "call"},
        {"case": "expired option", "market_price": 5.0, "S": 100.0, "K": 100.0, "T": 0.0, "r": 0.04, "option_type": "call"},
        {"case": "above upper bound", "market_price": 150.0, "S": 100.0, "K": 100.0, "T": 0.5, "r": 0.04, "option_type": "call"},
    ]
)

failure_rows = []

for _, row in failure_cases.iterrows():
    result = bs_c.implied_vol_bisection(
        row["market_price"],
        row["S"],
        row["K"],
        row["T"],
        row["r"],
        row["option_type"],
    )

    failure_rows.append(
        {
            "case": row["case"],
            "solver_status": result.solver_status,
            "failure_reason": result.failure_reason,
            "status_code": result.status_code,
        }
    )

failure_table = pd.DataFrame(failure_rows)
failure_table

## Fallback behavior

The safe wrapper uses the C solver when the shared library is available. If the library is missing or the C call fails, it falls back to the Python bisection solver.

In [ ]:
fallback_result = safe_c_implied_vol_bisection(
    market_price=comparison.loc[0, "market_price"],
    S=100.0,
    K=100.0,
    T=0.5,
    r=0.04,
    option_type="call",
    library_path=project_root / "compiled" / "missing_library.so",
)

{
    "implied_volatility": fallback_result.implied_volatility,
    "solver_status": fallback_result.solver_status,
    "used_fallback": fallback_result.used_fallback,
}

## Small benchmark

This benchmark compares repeated scalar IV solves. A scalar `ctypes` call includes Python-to-C boundary overhead, so the result should be treated as a local measurement rather than a broad performance claim.

In [ ]:
market_price = call_price(100.0, 100.0, 0.5, 0.04, 0.25)
iterations = 5_000

python_time = timeit.timeit(
    lambda: solve_implied_volatility(market_price, 100.0, 100.0, 0.5, 0.04, "call"),
    number=iterations,
)

c_time = timeit.timeit(
    lambda: bs_c.implied_vol_bisection(market_price, 100.0, 100.0, 0.5, 0.04, "call"),
    number=iterations,
)

benchmark = pd.DataFrame(
    {
        "implementation": ["Python bisection", "C bisection through ctypes"],
        "iterations": [iterations, iterations],
        "total_seconds": [python_time, c_time],
        "milliseconds_per_solve": [
            python_time / iterations * 1_000,
            c_time / iterations * 1_000,
        ],
    }
)

benchmark.round(6)

## Accuracy note

The C and Python solvers should agree within tolerance before any performance discussion. For this project, accuracy and transparent failure handling matter more than claiming speed improvements.